# CommGuard detector evaluation v2

Restore an immutable benign archive, print coverage first, and fit only after the strict eight-family/three-run/30-second gate passes. The communication-only block is primary; other ablations are diagnostic.


In [ ]:
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_detector_evaluation_v2"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
REVIEWED_COMMIT = ""  # Required: immutable 40-character commit visible on origin.
REPOSITORY = Path("/kaggle/working/commguard-source")

if not re.fullmatch(r"[0-9a-f]{40}", REVIEWED_COMMIT):
    raise RuntimeError("Set REVIEWED_COMMIT to the reviewed, pushed 40-character commit SHA.")
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )
if not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin", REVIEWED_COMMIT], check=True)
subprocess.run(
    ["git", "-C", str(REPOSITORY), "checkout", "--detach", REVIEWED_COMMIT], check=True
)
head = subprocess.run(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPOSITORY), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
pushed_refs = subprocess.run(
    ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if head != REVIEWED_COMMIT or dirty or not pushed_refs:
    raise RuntimeError(
        "Reproducibility gate failed: "
        f"head={head} dirty={bool(dirty)} pushed={bool(pushed_refs)}"
    )
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
        "-e", str(REPOSITORY),
    ],
    check=True,
)
print({"reviewed_commit": head, "remote_refs": pushed_refs.splitlines()})


In [ ]:
from commguard.artifacts import restore_archive, sha256_file

INPUT_ARCHIVE = Path("/kaggle/input/commguard-benign-corpus-v2/commguard-benign-corpus-v2-REPLACE.tar.gz")
EXPECTED_INPUT_SHA256 = ""  # Required: SHA-256 printed by the preceding notebook.
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_INPUT_SHA256):
    raise RuntimeError("Set EXPECTED_INPUT_SHA256 to the exact 64-character archive hash.")
actual_input_sha256 = sha256_file(INPUT_ARCHIVE)
if actual_input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        "Input archive hash mismatch: "
        f"expected={EXPECTED_INPUT_SHA256} actual={actual_input_sha256}"
    )
restore_archive(INPUT_ARCHIVE, ARTIFACTS, expected_sha256=EXPECTED_INPUT_SHA256)
print({"restored_archive": str(INPUT_ARCHIVE), "sha256": actual_input_sha256})


In [ ]:
from datetime import datetime, timezone

from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CONTEXT = ProvenanceContext.create(
    corpus_id=f"corpus-detector-v2-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-detector-v2-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-detector-v2-{NOTEBOOK_RUN_ID}",
    notebook_version="commguard_detector_evaluation_v2",
    input_archive_sha256=EXPECTED_INPUT_SHA256,
    random_seed=20260730,
    repository_root=REPOSITORY,
)
if CONTEXT.source_dirty or CONTEXT.source_commit != REVIEWED_COMMIT:
    raise RuntimeError("SDK provenance no longer matches the clean reviewed source commit.")
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
import json
from collections import Counter

from commguard.features import (
    PRIMARY_BENIGN_FAMILIES,
    load_extraction_result,
    require_primary_coverage,
)

BENIGN_MATRIX_SUMMARY_PATH = Path("results/matrix-REPLACE.json")
if BENIGN_MATRIX_SUMMARY_PATH.is_absolute() or ".." in BENIGN_MATRIX_SUMMARY_PATH.parts:
    raise RuntimeError("Benign matrix summary path must be artifact-root-relative.")
matrix_summary_path = ARTIFACTS / BENIGN_MATRIX_SUMMARY_PATH
if not matrix_summary_path.is_file():
    raise RuntimeError(f"Exact benign matrix summary is missing: {BENIGN_MATRIX_SUMMARY_PATH}")
BENIGN_MATRIX_SUMMARY = json.loads(matrix_summary_path.read_text(encoding="utf-8"))
BENIGN_EXTRACTION_SUMMARY = Path(BENIGN_MATRIX_SUMMARY["feature_extraction_summary"])
BENIGN_EXTRACTION = load_extraction_result(ARTIFACTS, BENIGN_EXTRACTION_SUMMARY)
if BENIGN_EXTRACTION.calibration_reference != BENIGN_MATRIX_SUMMARY["calibration_reference"]:
    raise RuntimeError("Benign extraction and matrix calibration references differ.")
coverage_by_family = {}
for family in PRIMARY_BENIGN_FAMILIES:
    records = [record for record in BENIGN_EXTRACTION.coverage if record.workload_family == family]
    coverage_by_family[family] = {
        "planned": len(records),
        "included": sum(record.status == "included" for record in records),
        "reason_counts": dict(
            Counter(record.reason_code for record in records if record.reason_code)
        ),
    }
for family, row in sorted(coverage_by_family.items()):
    print({"family": family, **row})
COVERAGE_GATE = require_primary_coverage(
    BENIGN_EXTRACTION,
    required_families=PRIMARY_BENIGN_FAMILIES,
    minimum_runs_per_family=3,
)
print({"coverage_gate": COVERAGE_GATE})


In [ ]:
from commguard.evaluation import evaluate_detector

RUN_DETECTOR_EVALUATION = True
if not RUN_DETECTOR_EVALUATION:
    raise RuntimeError("Detector evaluation was disabled after the coverage gate.")
EVALUATION = evaluate_detector(
    input_root=ARTIFACTS,
    output=ARTIFACTS,
    required_families=PRIMARY_BENIGN_FAMILIES,
    minimum_runs_per_family=3,
    benign_extraction_summary=BENIGN_EXTRACTION_SUMMARY.relative_to(ARTIFACTS),
)
print({
    "primary_communication_only": EVALUATION["primary_communication_only"],
    "coverage_gate": EVALUATION["coverage_gate"],
    "warnings": EVALUATION["warnings"],
    "exact_calibration_reference": EVALUATION["calibration_reference"],
    "evaluation_artifact_for_next_notebook": EVALUATION["result_artifact"],
})


## Results

not executed. No accuracy, robustness, or generalization claim is present.


In [ ]:
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-detector-evaluation-v2-{NOTEBOOK_RUN_ID}.tar.gz")
ArtifactStore(ARTIFACTS).export(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
SHA_FILE = ARCHIVE.with_suffix(ARCHIVE.suffix + ".sha256")
SHA_FILE.write_text(f"{ARCHIVE_SHA256}  {ARCHIVE.name}\n", encoding="utf-8")
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(f"NEXT STEP: copy SHA-256 {ARCHIVE_SHA256} into EXPECTED_INPUT_SHA256 in commguard_adversarial_redteam_v1.ipynb.")
print(
    f"NEXT STEP: set that notebook's REVIEWED_COMMIT to {REVIEWED_COMMIT} "
    "and run from the first cell."
)
